# TP 3 — Détecter des anomalies avec des méthodes statistiques
## Séance 3 : Z-score, score robuste, IQR et choix du seuil
### Version corrigée (enseignant)

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

Vous reprenez le jeu de **10 000 transactions** nettoyé lors du TP 2. Vous allez tester trois méthodes statistiques pour détecter les transactions frauduleuses à partir du **montant**, puis mesurer leurs limites.

1. Z-score classique ;
2. score robuste (médiane et MAD) ;
3. écart interquartile (IQR), sur le montant puis sur son logarithme ;
4. score par catégorie (anomalies contextuelles) ;
5. influence du seuil sur la précision et le rappel ;
6. limite de l’approche univariée : la distance de Mahalanobis.

**Rappel des formules**

- Z-score : z = (x − μ) / σ ; alerte si |z| > 3.
- Z robuste : 0,6745 × (x − médiane) / MAD, avec MAD = médiane des |x − médiane| ; alerte si |z robuste| > 3,5.
- IQR : alerte si x < Q1 − 1,5 × IQR ou x > Q3 + 1,5 × IQR, avec IQR = Q3 − Q1.

## Étape 1 — Données et fonction d’évaluation (fournies)

Exécutez la cellule suivante **sans la modifier**. Elle recrée le jeu de données, applique le nettoyage du TP 2 et définit la fonction `evaluer(y, alerte)`, qui affiche la matrice de confusion, la précision et le rappel.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def generer_transactions(seed=7):
    rng = np.random.default_rng(seed)
    n = 10000
    n_clients = 500

    poids = np.array([1] * 6 + [4] * 16 + [2] * 2, dtype=float)
    poids /= poids.sum()

    categories = ['alimentation', 'restauration', 'transport', 'vetements',
                  'electronique', 'voyage', 'jeux_en_ligne']
    p_cat = [0.30, 0.22, 0.15, 0.12, 0.08, 0.06, 0.07]
    mu = {'alimentation': 3.3, 'restauration': 3.2, 'transport': 2.8, 'vetements': 4.0,
          'electronique': 5.0, 'voyage': 5.3, 'jeux_en_ligne': 3.0}

    categorie = rng.choice(categories, size=n, p=p_cat)
    montant = np.array([rng.lognormal(mu[c], 0.5) for c in categorie])
    heure = rng.choice(24, size=n, p=poids)
    canal = rng.choice(['carte_presente', 'sans_contact', 'en_ligne'], size=n, p=[0.45, 0.30, 0.25])
    canal[categorie == 'jeux_en_ligne'] = 'en_ligne'
    pays = rng.choice(['FR', 'ES', 'IT', 'DE', 'US', 'NG'], size=n, p=[0.90, 0.03, 0.03, 0.02, 0.015, 0.005])
    fraude = np.zeros(n, dtype=int)

    # 100 fraudes : la nuit, en ligne, à l'étranger, montants élevés
    idx = rng.choice(n, size=100, replace=False)
    fraude[idx] = 1
    heure[idx] = np.where(rng.random(100) < 0.8, rng.integers(0, 6, 100), rng.integers(6, 24, 100))
    categorie[idx] = rng.choice(['electronique', 'voyage', 'jeux_en_ligne'], size=100, p=[0.4, 0.3, 0.3])
    montant[idx] = rng.lognormal(5.2, 0.6, 100)
    canal[idx] = np.where(rng.random(100) < 0.9, 'en_ligne', 'carte_presente')
    pays[idx] = np.where(rng.random(100) < 0.6, rng.choice(['US', 'NG', 'ES', 'DE'], size=100), 'FR')

    jour = rng.integers(0, 90, n)
    minute = rng.integers(0, 60, n)
    date_heure = (pd.Timestamp('2026-01-01') + pd.to_timedelta(jour, unit='D')
                  + pd.to_timedelta(heure, unit='h') + pd.to_timedelta(minute, unit='m'))

    df = pd.DataFrame({
        'id_transaction': [f'T{i:05d}' for i in range(n)],
        'id_client': [f'C{c:03d}' for c in rng.integers(0, n_clients, n)],
        'date_heure': date_heure,
        'montant': montant.round(2),
        'categorie': categorie,
        'canal': canal,
        'pays': pays,
        'fraude': fraude,
    })

    # Achats exceptionnels légitimes (anomalies réelles, mais pas des fraudes)
    legit = rng.choice(df.index[df['fraude'] == 0], size=3, replace=False)
    df.loc[legit, 'montant'] = [3200.00, 4500.00, 5800.00]
    df.loc[legit, 'categorie'] = 'voyage'

    # Problèmes de qualité : erreurs de saisie, valeurs négatives
    non_fraude = df.index[df['fraude'] == 0].to_numpy()
    saisie = rng.choice(non_fraude, size=4, replace=False)
    df.loc[saisie, 'montant'] = 99999.99
    neg = rng.choice(np.setdiff1d(non_fraude, saisie), size=6, replace=False)
    df.loc[neg, 'montant'] = -df.loc[neg, 'montant']

    # Valeurs manquantes
    df.loc[rng.choice(n, size=100, replace=False), 'montant'] = np.nan
    df.loc[rng.choice(n, size=200, replace=False), 'canal'] = np.nan

    # Doublons exacts
    doublons = df.sample(40, random_state=1)
    df = pd.concat([df, doublons], ignore_index=True)
    df = df.sort_values('date_heure', kind='stable').reset_index(drop=True)
    return df

# Données du TP 2, nettoyées
df = generer_transactions().drop_duplicates()
df.loc[(df['montant'] <= 0) | (df['montant'] >= 99999), 'montant'] = np.nan
df['montant'] = df['montant'].fillna(df.groupby('categorie')['montant'].transform('median'))
df['canal'] = df['canal'].fillna('inconnu')
df['heure'] = df['date_heure'].dt.hour
df = df.reset_index(drop=True)

def evaluer(y, alerte, afficher=True):
    alerte = np.asarray(alerte).astype(int)
    y = np.asarray(y)
    tp = int(((alerte == 1) & (y == 1)).sum())
    fp = int(((alerte == 1) & (y == 0)).sum())
    fn = int(((alerte == 0) & (y == 1)).sum())
    tn = int(((alerte == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    rappel = tp / (tp + fn)
    if afficher:
        print(f'Alertes={tp + fp}  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
        print(f'Précision={precision:.2f}  Rappel={rappel:.2f}')
    return precision, rappel

print(df.shape, 'dont', df['fraude'].sum(), 'fraudes')

## Étape 2 — Le Z-score classique

Complétez la cellule (remplacez chaque `...`). Utilisez la colonne `montant`.

In [ ]:
mu = df['montant'].mean()
sigma = df['montant'].std()
df['z'] = (df['montant'] - mu) / sigma
print(f'mu = {mu:.2f} €   sigma = {sigma:.2f} €   seuil haut (mu + 3 sigma) = {mu + 3 * sigma:.1f} €')

alerte_z = (df['z'].abs() > 3).astype(int)      # 1 si |z| > 3
evaluer(df['fraude'], alerte_z)

## Étape 3 — Le score robuste (médiane et MAD)

In [ ]:
mediane = df['montant'].median()
mad = (df['montant'] - mediane).abs().median()           # médiane des |x - médiane|
df['z_rob'] = 0.6745 * (df['montant'] - mediane) / mad   # 0.6745 * (x - médiane) / MAD
print(f'médiane = {mediane:.2f} €   MAD = {mad:.2f} €')

alerte_rob = (df['z_rob'].abs() > 3.5).astype(int)       # 1 si |z_rob| > 3.5
evaluer(df['fraude'], alerte_rob)

## Étape 4 — L’écart interquartile (IQR)

Appliquez la règle de Tukey (1,5 × IQR) d’abord sur le montant, puis sur le **logarithme** du montant.

In [ ]:
# IQR sur le montant
q1 = df['montant'].quantile(0.25)
q3 = df['montant'].quantile(0.75)
iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr
print(f'Q1 = {q1:.1f}   Q3 = {q3:.1f}   IQR = {iqr:.1f}   bornes = [{borne_basse:.1f} ; {borne_haute:.1f}]')

alerte_iqr = ((df['montant'] < borne_basse) | (df['montant'] > borne_haute)).astype(int)
evaluer(df['fraude'], alerte_iqr)

In [ ]:
# IQR sur le logarithme du montant
df['log_montant'] = np.log(df['montant'])
q1l = df['log_montant'].quantile(0.25)
q3l = df['log_montant'].quantile(0.75)
iqrl = q3l - q1l
alerte_iqr_log = ((df['log_montant'] < q1l - 1.5 * iqrl) | (df['log_montant'] > q3l + 1.5 * iqrl)).astype(int)
print('Bornes en euros :', round(np.exp(q1l - 1.5 * iqrl), 1), 'à', round(np.exp(q3l + 1.5 * iqrl), 1))
evaluer(df['fraude'], alerte_iqr_log)

## Étape 5 — Un score par catégorie (anomalies contextuelles)

Un montant n’est pas élevé « en soi » mais pour une catégorie de commerce. Calculez le score robuste **à l’intérieur de chaque catégorie**, sur le logarithme du montant.

In [ ]:
g = df.groupby('categorie')['log_montant']
mediane_cat = g.transform('median')
mad_cat = g.transform(lambda s: (s - s.median()).abs().median())

df['z_cat'] = 0.6745 * (df['log_montant'] - mediane_cat) / mad_cat
alerte_cat = (df['z_cat'].abs() > 3.5).astype(int)
evaluer(df['fraude'], alerte_cat)

**Fraudes détectées par catégorie**

In [ ]:
# Pour chaque catégorie : nombre de fraudes détectées et nombre total de fraudes
fraudes = df[df['fraude'] == 1].assign(detectee=alerte_cat[df['fraude'] == 1])
fraudes.groupby('categorie')['detectee'].agg(['sum', 'count'])

## Étape 6 — Influence du seuil

Faites varier le seuil du score par catégorie et observez la précision et le rappel.

In [ ]:
seuils = [2, 2.5, 3, 3.5, 4, 5, 6]
precisions, rappels = [], []
for s in seuils:
    p, r = evaluer(df['fraude'], (df['z_cat'].abs() > s), afficher=False)
    precisions.append(p)
    rappels.append(r)

resultats = pd.DataFrame({'seuil': seuils, 'precision': precisions, 'rappel': rappels,
                          'alertes': [(df['z_cat'].abs() > s).sum() for s in seuils]})
resultats.round(2)

In [ ]:
# Courbes précision et rappel en fonction du seuil
plt.figure(figsize=(8, 3.5))
plt.plot(seuils, precisions, marker='o', label='Précision')
plt.plot(seuils, rappels, marker='o', label='Rappel')
plt.xlabel('Seuil sur |z_cat|')
plt.ylabel('Précision / rappel')
plt.legend()
plt.show()

## Étape 7 — La limite de l’approche univariée

Dans ce jeu de 503 clients, le revenu mensuel et les dépenses par carte sont corrélés. Les **3 derniers clients** ont un comportement atypique : leurs valeurs, prises une à une, paraissent normales. Exécutez la cellule suivante pour créer les données.

In [ ]:
rng = np.random.default_rng(0)
n = 500
revenu = rng.normal(3000, 800, n)
depense = 0.4 * revenu + rng.normal(0, 100, n)
donnees = pd.DataFrame({'revenu': revenu.round(0), 'depense': depense.round(0)})

# 3 clients atypiques ajoutés à la fin
donnees.loc[len(donnees)] = [2300, 1700]
donnees.loc[len(donnees)] = [3800, 600]
donnees.loc[len(donnees)] = [1800, 1300]

vraies_anomalies = np.arange(len(donnees)) >= 500
print(donnees.shape, '| corrélation revenu / dépenses :', round(donnees['revenu'].corr(donnees['depense']), 2))

**a) Z-scores univariés**

In [ ]:
# a) Z-scores univariés sur chaque colonne
z = (donnees - donnees.mean()) / donnees.std()
alerte_uni = (z.abs() > 3).any(axis=1).to_numpy()
print('Alertes univariées :', alerte_uni.sum(), '| vraies anomalies détectées :', (alerte_uni & vraies_anomalies).sum(), 'sur 3')
z.tail(3).round(2)

**b) Distance de Mahalanobis**

Formule : D² = (x − μ)ᵀ Σ⁻¹ (x − μ). Pour 2 variables, D² suit approximativement une loi du χ² à 2 degrés de liberté : le seuil à 99,7 % vaut −2 ln(0,003) ≈ 11,6.

In [ ]:
# b) Distance de Mahalanobis
X = donnees[['revenu', 'depense']].to_numpy()
mu_vec = X.mean(axis=0)                       # vecteur des moyennes
cov = np.cov(X, rowvar=False)                 # matrice de covariance
cov_inv = np.linalg.inv(cov)                  # inverse
diff = X - mu_vec
d2 = np.einsum('ij,jk,ik->i', diff, cov_inv, diff)    # D² de chaque client

seuil_d2 = -2 * np.log(0.003)                 # seuil à 99,7 % pour 2 variables
alerte_maha = d2 > seuil_d2
print(f'Seuil D² = {seuil_d2:.2f}')
print('Alertes Mahalanobis :', alerte_maha.sum(), '| vraies anomalies détectées :', (alerte_maha & vraies_anomalies).sum(), 'sur 3')
print('D² des 3 clients atypiques :', d2[-3:].round(1))

**c) Visualisation**

In [ ]:
# c) Nuage de points : clients normaux, clients atypiques, alertes Mahalanobis
plt.figure(figsize=(7, 4.5))
plt.scatter(donnees['revenu'][~vraies_anomalies], donnees['depense'][~vraies_anomalies], s=10, alpha=0.5, label='Clients normaux')
plt.scatter(donnees['revenu'][vraies_anomalies], donnees['depense'][vraies_anomalies], s=60, color='red', label='Clients atypiques')
plt.scatter(donnees['revenu'][alerte_maha], donnees['depense'][alerte_maha], s=120, facecolors='none', edgecolors='black', label='Alertes Mahalanobis')
plt.xlabel('Revenu mensuel (€)')
plt.ylabel('Dépenses par carte (€ / mois)')
plt.legend()
plt.show()

## Questions

Répondez dans les cellules de texte situées sous chaque question.

**Question 1.** Quel est le seuil μ + 3σ du Z-score classique ? Combien de fraudes détecte-t-il ? Pourquoi si peu ?

**Réponse 1.** μ ≈ 57,2 € et σ ≈ 108,7 €, donc le seuil haut vaut μ + 3σ ≈ **383 €**. Le Z-score signale 87 transactions : 11 fraudes détectées et 76 fausses alertes (précision 0,13, rappel 0,11) ; **89 fraudes sur 100 sont manquées**. Deux raisons : le montant médian d’une fraude (environ 190 €) est bien inférieur à 383 €, et σ est **gonflé par les montants extrêmes**. En retirant les 3 achats légitimes de plus de 2 000 €, σ tombe à environ 75 € et le seuil à environ 280 € : les extrêmes masquent les anomalies, comme vu en cours.

**Question 2.** Comparez le score robuste et l’IQR sur le montant brut avec le Z-score classique (nombre d’alertes, précision, rappel). Pourquoi y a-t-il autant de fausses alertes ?

**Réponse 2.** Le score robuste (borne haute ≈ 98 €) donne 1 439 alertes : 87 fraudes détectées, 1 352 fausses alertes (précision 0,06, rappel 0,87). L’IQR donne 1 324 alertes (borne haute ≈ 106 €) : 81 fraudes détectées, précision 0,06, rappel 0,81. Ces méthodes détectent beaucoup plus de fraudes que le Z-score, mais avec environ **15 fausses alertes pour chaque vraie fraude**. La raison : la distribution des montants est très asymétrique, et de nombreuses transactions légitimes sont élevées parce qu’elles concernent des catégories chères (électronique, voyage). Le montant brut ne tient pas compte de la catégorie.

**Question 3.** Quel est l’effet du passage au logarithme avec l’IQR (nombre d’alertes, précision, rappel) ?

**Réponse 3.** Avec le logarithme, les bornes deviennent environ 3,9 € et 259 €. On passe à 338 alertes (3,4 %) : 29 fraudes détectées, précision 0,09, rappel 0,29. Le nombre de fausses alertes est divisé par 4 environ, mais on perd beaucoup de fraudes. Le passage au log rend la distribution plus symétrique, mais ne suffit pas : le problème de fond (les catégories ont des niveaux de montant différents) reste.

**Question 4.** Que change le calcul du score par catégorie ? Quelles fraudes sont détectées et lesquelles ne le sont pas ? Pourquoi ?

**Réponse 4.** Avec le score robuste par catégorie (seuil 3,5), on obtient 31 alertes : 24 fraudes détectées et 7 fausses alertes (précision 0,77, rappel 0,24). La précision est bien meilleure, car chaque montant est comparé aux montants normaux de **sa** catégorie. Les 24 fraudes détectées sont toutes en **jeux en ligne** (sur 35) : un montant de plusieurs centaines d’euros y est très inhabituel (médiane d’environ 21 €). En revanche, **aucune des fraudes en électronique (31) ni en voyage (34) n’est détectée** : leurs montants sont plausibles pour ces catégories (médianes d’environ 146 € et 193 €). Le montant seul ne permet pas de les distinguer.

**Question 5.** Observez le tableau précision / rappel selon le seuil. Quel seuil choisiriez-vous pour une équipe d’analystes qui peut traiter une soixantaine d’alertes sur ces 10 000 transactions ? Justifiez.

**Réponse 5.**

| Seuil | Alertes | Précision | Rappel |
|---|---|---|---|
| 2 | 505 | 0,07 | 0,36 |
| 2,5 | 169 | 0,20 | 0,33 |
| 3 | 64 | 0,47 | 0,30 |
| 3,5 | 31 | 0,77 | 0,24 |
| 4 | 22 | 0,86 | 0,19 |
| 5 | 12 | 0,75 | 0,09 |
| 6 | 2 | 0,50 | 0,01 |

Avec une soixantaine d’alertes à traiter, le **seuil 3** (64 alertes) est le choix naturel : précision 0,47 et rappel 0,30, soit presque une alerte sur deux qui est une vraie fraude et environ un tiers des fraudes détectées. Le seuil 3,5 est plus précis mais ne tient que la moitié de la capacité de l’équipe ; les seuils plus bas noieraient les analystes de fausses alertes. D’autres réponses sont acceptables si elles s’appuient sur un critère (capacité de traitement, coût d’une fraude manquée, coût d’une fausse alerte).

**Question 6.** Pourquoi les Z-scores univariés ne détectent-ils pas les 3 clients atypiques alors que la distance de Mahalanobis les détecte ? Qu’est-ce que cela implique pour la détection de fraude à partir du seul montant ?

**Réponse 6.** Pris isolément, le revenu et les dépenses des 3 clients atypiques sont proches de la moyenne (leurs Z-scores restent entre −1,7 et 1,5, bien en dessous de 3). Les Z-scores univariés ne détectent aucune de ces 3 anomalies et signale à la place 4 clients normaux situés dans les queues des distributions. L’anomalie est dans la **combinaison** : le revenu et les dépenses sont très corrélés (environ 0,94), et ces clients s’écartent de cette relation. La distance de Mahalanobis, qui tient compte de la covariance, leur attribue des D² de 50, 68 et 29, très au-dessus du seuil de 11,6 : les 3 sont détectés (avec 2 fausses alertes).

Pour la fraude, cela montre que le montant seul ne suffit pas : il faut combiner plusieurs variables (heure, pays, canal, historique du client), ce que feront les méthodes des séances suivantes.

## Pour aller plus loin (facultatif)

1. Retirez du jeu les 3 achats légitimes de plus de 2 000 € et recalculez le Z-score classique. Comment évoluent μ, σ et le seuil ? Que dit ce résultat du masquage ?
2. Calculez le score robuste par catégorie sur le montant brut (et non sur son logarithme). Les résultats sont-ils meilleurs ou moins bons ? Pourquoi ?
3. Combinez deux règles : alerte si |z_cat| > 3 **ou** si la transaction a lieu entre 0 h et 6 h avec un montant supérieur à 100 €. Comparez la précision et le rappel.

## Pistes pour « Pour aller plus loin »

- Sans les 3 achats de plus de 2 000 €, σ tombe d’environ 109 € à environ 75 € et le seuil μ + 3σ d’environ 383 € à environ 280 € : trois valeurs sur 10 000 suffisent à relever nettement le seuil. C’est le masquage.
- Sur le montant brut, la distribution par catégorie reste asymétrique : le log donne en général des résultats plus stables. Vérifiez-le en comparant les nombres d’alertes.
- Le critère « nuit et montant > 100 € » cible les fraudes de nuit que le montant seul manquait, notamment en électronique et en voyage : le rappel augmente, au prix de quelques fausses alertes. C’est la motivation du *feature engineering* et des modèles multivariés (séances 4 à 8).